# SO-101 Perception Benchmark — SAM 3 + metric-depth vs known ground truth

Runs the **`find_object` perception components** on simulation frames whose object
centre is known **exactly**, and scores each against ground truth so we pick the
pipeline on measured error, not vibes.

**Two things measured, separately:**
1. **Detection (SAM 3)** — open-vocab text prompt (`"red cube"`) → predicted centre pixel.
   Scored vs the exact projected pixel (pixel error + detection rate).
2. **Metric depth** (Depth Pro / UniDepthV2 / Metric3D-v2) — depth at a pixel → back-project
   with `camera_math.point_at_depth` → world (x,y,z). Scored vs the exact world centre, both
   at the **GT pixel** (isolates depth) and at the **SAM pixel** (full pipeline).

## Honesty
Ground truth is used **only** to (a) know which pixel is the true centre and (b) score error.
No GT drives any estimate. Every depth backend must return depth in **metres along the camera
OPTICAL / FORWARD axis** — the convention `camera_math.point_at_depth` consumes.

## Workflow
Clone this repo on Colab and run top-to-bottom. Depth backends have conflicting deps — run
**one at a time** (install cell per backend). When something breaks, tell the author which
cell + error; they push a fix and you `!git pull`.

## 0. Setup — clone/cd + imports

In [ ]:
# On Colab: clone the repo (set CLONE_URL after first push) and cd into it.
import os, sys
CLONE_URL = 'https://github.com/Yunsmn/RoboticsPerceptionTest.git'
if not os.path.exists('camera_math.py'):
    repo = CLONE_URL.rstrip('/').split('/')[-1].removesuffix('.git')
    if not os.path.exists(repo):
        os.system(f'git clone {CLONE_URL}')
    os.chdir(repo)
sys.path.insert(0, '.')
import numpy as np, json, math
from pathlib import Path
import camera_math as CM      # pure-numpy; standalone copy of the sim's camera model
print('camera_math OK:', np.round(CM.side_cam_params()[:3], 2))

## 1. Install a backend (run ONE per session — deps conflict)

In [ ]:
# Uncomment exactly ONE, run it, then run its adapter + eval below.
# !pip -q install ultralytics                 # SAM 3 (detection)
# !pip -q install git+https://github.com/apple/ml-depth-pro.git   # Depth Pro
# !pip -q install unidepth                    # UniDepthV2
# !pip -q install mmcv mmengine timm          # Metric3D-v2 (torch.hub)

## 2. Data — sim frames with an EXACTLY known centre

`data/manifest.json` (from the sim harness) + `data/frames/pose_XXX.png`. Each frame carries
`gt_xyz` (world metres, arm-base frame), `gt_uv` (the exact projected centre pixel, self-checked
to 1e-16 m) and `gt_depth_m`. The camera intrinsics/extrinsics are in `manifest['camera']`.

In [ ]:
man = json.loads(Path('data/manifest.json').read_text())
cam = man['camera']
f, cx, cy = cam['f'], cam['cx'], cam['cy']
cam_pos = np.array(cam['cam_pos'], float)
R_cw    = np.array(cam['R_cw'], float)
W, H = cam['width'], cam['height']
FRAMES = man['frames']
print(len(FRAMES), 'frames |', f'{W}x{H}', '| f=%.1f' % f)

# HERO frame for the single-image demo (nearest the workspace centre).
hero = min(FRAMES, key=lambda fr: abs(fr['gt_xyz'][1]))
print('hero:', hero['png'], '| exact centre world', np.round(hero['gt_xyz'],4),
      '| exact centre pixel', hero['gt_uv'])

## 3. Camera model — project the known centre to its pixel
Inverse of `point_at_depth`: world → (u, v, forward-axis depth). Same convention the harness used.

In [ ]:
def project(P):
    '''World point -> (u, v, forward_axis_depth_m).'''
    xc, yc, zc = R_cw.T @ (np.asarray(P, float) - cam_pos)
    depth = -zc
    return cx + f * xc / depth, cy - f * yc / depth, depth

# sanity: our own projection must reproduce the manifest's stored gt_uv
u,v,_ = project(hero['gt_xyz'])
print('reproj vs manifest gt_uv:', np.round([u,v],3), 'vs', hero['gt_uv'])

## 4. SAM 3 detection adapter
Open-vocab text prompt → the object's centre pixel. Uses the ultralytics `SAM` interface
(auto-downloads `sam3.pt`). **If the API differs in your ultralytics build, this is the cell
to tell the author about** — it's best-effort.

In [ ]:
_sam = None
def sam3_center(img_rgb, prompt='red cube'):
    '''Return (u, v) of the best instance matching `prompt`, or None.'''
    global _sam
    from ultralytics import SAM
    if _sam is None:
        _sam = SAM('sam3.pt')
    # best-effort: ultralytics SAM3 takes a text/concept prompt; adjust kw if needed.
    res = _sam(img_rgb, texts=[prompt], verbose=False)
    r = res[0]
    if r.masks is None or len(r.masks) == 0:
        return None
    # pick the highest-confidence instance; centroid of its mask
    m = r.masks.data.cpu().numpy()
    conf = r.boxes.conf.cpu().numpy() if r.boxes is not None else np.ones(len(m))
    best = m[int(np.argmax(conf))] > 0.5
    ys, xs = np.where(best)
    return (float(xs.mean()), float(ys.mean())) if len(xs) else None

### 4a. SAM 3 detection error over all frames

In [ ]:
from PIL import Image
def eval_sam(prompt='red cube'):
    px_err, miss = [], 0
    for fr in FRAMES:
        img = np.array(Image.open('data/'+fr['png']).convert('RGB'))
        c = sam3_center(img, prompt)
        if c is None:
            miss += 1; continue
        gu, gv = fr['gt_uv']
        px_err.append(math.hypot(c[0]-gu, c[1]-gv))
    e = np.array(px_err) if px_err else np.array([np.nan])
    return {'n': len(FRAMES), 'detected': len(px_err), 'miss': miss,
            'px_err_mean': float(np.nanmean(e)), 'px_err_median': float(np.nanmedian(e))}
# print(eval_sam())   # run after installing ultralytics

## 5. Metric-depth adapters
Each returns an `(H, W)` float32 map of **forward-axis metric depth (metres)**. Convert range /
disparity / z as needed. Best-effort against each model's API — **the cells to iterate on**.

In [ ]:
def infer_depth_pro(img_rgb):
    import depth_pro, torch
    model, tf = depth_pro.create_model_and_transforms(); model.eval()
    with torch.no_grad():
        pred = model.infer(tf(img_rgb), f_px=f)
    return pred['depth'].squeeze().cpu().numpy().astype('float32')   # metric metres

def infer_unidepth_v2(img_rgb):
    import torch; from unidepth.models import UniDepthV2
    model = UniDepthV2.from_pretrained('lpiccinelli/unidepth-v2-vitl14').eval()
    rgb = torch.from_numpy(img_rgb).permute(2,0,1)
    K = torch.tensor([[f,0,cx],[0,f,cy],[0,0,1]], dtype=torch.float32)
    with torch.no_grad():
        pred = model.infer(rgb, K)
    return pred['depth'].squeeze().cpu().numpy().astype('float32')   # metric z, metres

def infer_metric3d_v2(img_rgb):
    import torch
    model = torch.hub.load('yvanyin/metric3d', 'metric3d_vit_small', pretrain=True).cuda().eval()
    # NOTE: Metric3D needs its canonical-camera resize + intrinsic scaling. TODO: port the
    # official demo transform (intrinsic = [f,f,cx,cy]); this stub will need that to be metric.
    raise NotImplementedError('port Metric3D canonical transform before use')

BACKENDS = {'depth_pro': infer_depth_pro, 'unidepth_v2': infer_unidepth_v2,
            'metric3d_v2': infer_metric3d_v2}

## 6. Eval loop — loc error at the standoff
Depth at a pixel → `point_at_depth` → world → error vs known centre. Scored two ways:
**GT pixel** (isolates depth accuracy) and **SAM pixel** (full detection+depth pipeline).

In [ ]:
def eval_depth(infer, use_sam=False, prompt='red cube'):
    loc, zerr, n = [], [], 0
    for fr in FRAMES:
        img = np.array(Image.open('data/'+fr['png']).convert('RGB'))
        P_gt = np.array(fr['gt_xyz'], float)
        if use_sam:
            c = sam3_center(img, prompt)
            if c is None: continue
            u, v = c
        else:
            u, v = fr['gt_uv']
        depth = infer(img)
        d = float(depth[int(round(v)), int(round(u))])
        P = CM.point_at_depth(u, v, f, cx, cy, cam_pos, R_cw, d)
        loc.append(np.linalg.norm(P - P_gt)*1000); zerr.append(abs(P[2]-P_gt[2])*1000); n += 1
    L = np.array(loc)
    return {'n': n, 'loc_mean_mm': float(L.mean()), 'loc_median_mm': float(np.median(L)),
            'loc_p95_mm': float(np.percentile(L,95)), 'z_median_mm': float(np.median(zerr))}

In [ ]:
# Run per installed backend (GT-pixel first to isolate depth quality):
results = {}
for name, infer in BACKENDS.items():
    try:
        results[name] = {'at_gt_px': eval_depth(infer, use_sam=False)}
        # results[name]['full_pipeline'] = eval_depth(infer, use_sam=True)  # needs SAM installed
    except Exception as e:
        results[name] = {'error': f'{type(e).__name__}: {str(e)[:80]}'}
for k, v in results.items(): print(f'{k:14s}', v)

## 7. Compare to the pipelines we already measured
Our earlier SO-101 numbers (same sim, same ~1.15 m side-cam standoff, ~2 mm grasp basin):

| pipeline | detector | depth source | loc error | notes |
|---|---|---|---|---|
| HSV + plane | red threshold | known plane z=0.015 (prior) | ~2.0 mm | colour + plane priors |
| SAM + plane | FastSAM (colour-free) | known plane z=0.015 (prior) | ~1.7 mm | plane prior |
| mono SAM+depth | FastSAM | Depth-Anything (affine) | ~12 mm | honest, no plane |
| 3-cam triangulation | FastSAM | ray geometry | **1.1 / z 0.6 mm** | honest, multi-view |

The rows in `results` are the **new single-camera, plane-free** contenders. The bar:
**loc median (ideally p95) ≤ ~2 mm at the GT pixel** ⇒ single-shot metric depth can carry the
cube; else it positions coarsely and the wrist-camera servo closes the last mm (container is fine
at cm regardless).

In [ ]:
REFERENCE = {
    'hsv_plane':        {'loc_median_mm': 2.0,  'depth': 'known plane (prior)'},
    'sam_plane':        {'loc_median_mm': 1.7,  'depth': 'known plane (prior)'},
    'mono_sam_depth':   {'loc_median_mm': 12.0, 'depth': 'Depth-Anything affine'},
    'triangulation_3cam':{'loc_median_mm': 1.1, 'depth': 'multi-view geometry'},
}
print('--- reference (measured earlier) ---')
for k,v in REFERENCE.items(): print(f'  {k:20s} loc_median {v["loc_median_mm"]:5.1f} mm  ({v["depth"]})')
print('--- new contenders (this run) ---')
for k,v in results.items(): print(f'  {k:20s} {v}')

## 8. Single-image visualization — the known centre vs each estimate

In [ ]:
import matplotlib.pyplot as plt
img = np.array(Image.open('data/'+hero['png']).convert('RGB'))
gu, gv = hero['gt_uv']
plt.figure(figsize=(7,5)); plt.imshow(img)
plt.scatter([gu],[gv], marker='+', s=200, c='lime', label='exact centre (GT)')
# overlay a SAM detection if available:
try:
    c = sam3_center(img)
    if c: plt.scatter([c[0]],[c[1]], marker='x', s=120, c='red', label='SAM 3')
except Exception as e:
    print('SAM not run:', type(e).__name__)
plt.legend(); plt.title(f"{hero['png']}  |  GT world {np.round(hero['gt_xyz'],3)}"); plt.axis('off'); plt.show()

## 9. Decision & how to iterate
- **Winner** = lowest `loc_median_mm` at the GT pixel that also survives the SAM-pixel (full) run.
- Register it back in the main repo as a `MetricDepthBackend` + `PERCEPT_DEPTH_MODEL`.
- **Broken cell?** Tell the author the cell + exact error; they push, you `!git pull` and re-run.
- Regenerate/extend the dataset from the main repo:
  `MUJOCO_GL=egl venv/bin/python experiments/export_depth_benchmark_data.py --nx 5 --ny 9 --raised`.